# Compare Q3 2025 with last year

*Data Analysis in Python · Topic 6.3*

The quarterly review needs July to September 2025 compared three ways: each week with the week before, each week with the same week last year, and each month with the month before, with the conversion rate's change in points.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the `sessions` table (lesson 3.'s steps), and the cell that builds the `daily`, `weekly` and `monthly` tables from July 2024 to September 2025 (topics 6.1. and 6.2.'s steps): the `weekly` table has only full Monday-to-Sunday weeks, and both tables have a `conversion_rate` column.
2. In the first cell marked ✏️:
   - add two columns to the `weekly` table: `revenue_wow`, each week's revenue change from the week before, and `revenue_yoy`, its change from the same week a year earlier, both as fractions such as `0.12`
   - create the `q3_weeks` table: the rows of the `weekly` table from July to September 2025
   - create `best_yoy_week`: the label of the week in the `q3_weeks` table with the highest `revenue_yoy`
   - create `last_year_week`: the row of the `weekly` table for the week 52 weeks before `best_yoy_week`
3. In the second cell marked ✏️, add two columns to the `monthly` table: `revenue_mom`, each month's revenue change from the month before, as a fraction, and `rate_change`, its conversion rate's change from the month before, in percentage points (`0.0059` for 0.59 points).
4. In the last cell marked ✏️, create `q3_yoy`: July to September 2025's total revenue over July to September 2024's, minus 1, from the `daily` table.
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `q3_weeks` table has the quarter's 13 full weeks, from the week ending July 6 to the one ending September 28. `best_yoy_week` is the week ending September 28, 2025, with revenue up 58.6% on a year earlier and 17.9% on the week before. `last_year_week` is the week ending September 29, 2024, with $35,830.65 of revenue. In the `monthly` table, September 2025's `revenue_mom` is about `0.0854` and its `rate_change` about `0.0006`. `q3_yoy` is about `0.3765`.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = [f"{year}-{month:02d}" for year in (2024, 2025) for month in range(1, 13)][6:21]  # July 2024 to September 2025
START, END = "2024-07-01", "2025-09-30"

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Days, full weeks and months (topics 6.1 and 6.2). Run it, and leave it as it is.
daily = sessions.groupby("date")[["sessions", "transactions", "revenue"]].sum()

days_per_week = daily["sessions"].resample("W").count()
weekly = daily.resample("W").sum()
weekly = weekly[days_per_week == 7].copy()
weekly["conversion_rate"] = weekly["transactions"] / weekly["sessions"]

monthly = daily.resample("MS").sum()
monthly["conversion_rate"] = monthly["transactions"] / monthly["sessions"]
daily.shape, weekly.shape, monthly.shape

## Weeks

In [ ]:
weekly["revenue_wow"] = weekly["revenue"].pct_change()
weekly["revenue_yoy"] = weekly["revenue"].pct_change(52)
q3_weeks = weekly.loc["2025-07":"2025-09"]
best_yoy_week = q3_weeks["revenue_yoy"].idxmax()
last_year_week = weekly.loc[best_yoy_week - pd.Timedelta(weeks=52)]
q3_weeks, best_yoy_week, last_year_week

## Months

In [ ]:
monthly["revenue_mom"] = monthly["revenue"].pct_change()
monthly["rate_change"] = monthly["conversion_rate"].diff()
monthly.loc["2025-07":"2025-09"]

## The quarter

In [ ]:
q3_yoy = daily.loc["2025-07":"2025-09", "revenue"].sum() / daily.loc["2024-07":"2024-09", "revenue"].sum() - 1
q3_yoy

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    week = pd.Timestamp("2025-09-28")
    weekly = found.get("weekly")
    q3_weeks = found.get("q3_weeks")
    best = found.get("best_yoy_week")
    last_year = found.get("last_year_week")
    if not isinstance(weekly, pd.DataFrame) or len(weekly) != 65 or week not in weekly.index:
        results.append((False, "The weekly table should still be the provided cell's 65 full weeks. Run that cell as it is, and add columns to it."))
    elif not {"revenue_wow", "revenue_yoy"} <= set(weekly.columns):
        results.append((False, f"The weekly table should have the columns revenue_wow and revenue_yoy. It has {list(weekly.columns)}."))
    elif abs(weekly.loc[week, "revenue_wow"] - 1.179102) < 0.0001 or abs(weekly.loc[week, "revenue_yoy"] - 1.586413) < 0.0001:
        results.append((False, "The weekly table's changes are ratios, such as 1.18: this week over the other one, without the minus 1. pct_change() gives the change: 0.18."))
    elif abs(weekly.loc[week, "revenue_wow"] - 0.179102) > 0.0001:
        results.append((False, f"The weekly table's revenue_wow should be each week's change from the week before: weekly[\"revenue\"].pct_change(). For the week ending September 28, it's 0.1791, and yours is {weekly.loc[week, 'revenue_wow']!r}."))
    elif abs(weekly.loc[week, "revenue_yoy"] - 0.179102) < 0.0001:
        results.append((False, "The weekly table's revenue_yoy is the same as revenue_wow: it compares with the week before. Compare with the row 52 weeks earlier: pct_change(52)."))
    elif abs(weekly.loc[week, "revenue_yoy"] - 0.338278) < 0.0001:
        results.append((False, "The weekly table's revenue_yoy compares with the week 53 weeks earlier. A year of weeks is 52: pct_change(52)."))
    elif abs(weekly.loc[week, "revenue_yoy"] - 0.586413) > 0.0001 or weekly["revenue_yoy"].isna().sum() != 52:
        results.append((False, f"The weekly table's revenue_yoy should be each week's change from 52 weeks earlier: 0.5864 for the week ending September 28, and NaN for the first 52 weeks. It's {weekly.loc[week, 'revenue_yoy']!r}."))
    elif not isinstance(q3_weeks, pd.DataFrame) or len(q3_weeks) != 13 or q3_weeks.index[0] != pd.Timestamp("2025-07-06"):
        results.append((False, "q3_weeks should be the weekly table's 13 weeks of July to September 2025: weekly.loc[\"2025-07\":\"2025-09\"]."))
    elif best == pd.Timestamp("2025-07-06"):
        results.append((False, "best_yoy_week is the week with the most revenue. The brief asks for the highest revenue_yoy: q3_weeks[\"revenue_yoy\"].idxmax()."))
    elif best != week:
        results.append((False, f"best_yoy_week should be the label of the q3_weeks row with the highest revenue_yoy: q3_weeks[\"revenue_yoy\"].idxmax(). It is {best!r}."))
    elif not isinstance(last_year, pd.Series) or last_year.name != pd.Timestamp("2024-09-29"):
        results.append((False, "last_year_week should be the weekly table's row 52 weeks before best_yoy_week: weekly.loc[best_yoy_week - pd.Timedelta(weeks=52)]. A year back by the calendar is a Saturday, which isn't a week label."))
    elif abs(last_year["revenue"] - 35830.65) > 0.01:
        results.append((False, f"last_year_week's revenue should be $35,830.65. It's {last_year['revenue']!r}."))
    else:
        results.append((True, "The week ending September 28, 2025 grew most on the year before, 58.6%, against the week ending September 29, 2024: the same Monday to Sunday."))

    monthly = found.get("monthly")
    september = pd.Timestamp("2025-09-01")
    if not isinstance(monthly, pd.DataFrame) or september not in monthly.index or not {"revenue_mom", "rate_change"} <= set(monthly.columns):
        results.append((False, "The monthly table should be the provided cell's 15 months, with the columns revenue_mom and rate_change added."))
    elif abs(monthly.loc[september, "revenue_mom"] - 0.085412) > 0.0001:
        results.append((False, f"The monthly table's revenue_mom should be each month's change from the month before: monthly[\"revenue\"].pct_change(). September's is 0.0854, and yours is {monthly.loc[september, 'revenue_mom']!r}."))
    elif abs(monthly.loc[september, "rate_change"] - 0.027060) < 0.0001:
        results.append((False, "The monthly table's rate_change is the relative change, 2.7%. The brief asks for percentage points: monthly[\"conversion_rate\"].diff()."))
    elif abs(monthly.loc[september, "rate_change"] - 0.000612) > 0.000001:
        results.append((False, f"The monthly table's rate_change should be each month's conversion rate minus the month before's: 0.0006 in September. It's {monthly.loc[september, 'rate_change']!r}."))
    else:
        results.append((True, "September 2025's revenue is up 8.5% on August, and its conversion rate 0.06 points."))

    q3_yoy = found.get("q3_yoy")
    if not pd.api.types.is_number(q3_yoy) or isinstance(q3_yoy, (pd.DataFrame, pd.Series)):
        results.append((False, f"q3_yoy should be a number: the quarter's revenue total over the same quarter's a year earlier, minus 1. It is {q3_yoy!r}."))
    elif abs(q3_yoy - 0.388304) < 0.0001:
        results.append((False, "q3_yoy is 0.3883, the mean of the 13 weeks' changes. A quarter's change is its total over the other quarter's total, from the daily table."))
    elif abs(q3_yoy - 0.374650) < 0.0001:
        results.append((False, "q3_yoy is 0.3747, from the weekly table's totals. The weeks don't cover the quarters exactly: the week ending July 6, 2025 has days of June. Total the daily table's July to September."))
    elif abs(q3_yoy - 1.376501) < 0.0001:
        results.append((False, "q3_yoy is 1.38: the ratio, without the minus 1."))
    elif abs(q3_yoy - 0.376501) > 0.0001:
        results.append((False, f"q3_yoy should be July to September 2025's revenue total over July to September 2024's, minus 1: about 0.3765. It is {q3_yoy:.4f}."))
    else:
        results.append((True, f"July to September 2025's revenue is up {q3_yoy:.1%} on the same months of 2024."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
weekly["revenue_wow"] = weekly["revenue"].pct_change()
weekly["revenue_yoy"] = weekly["revenue"].pct_change(52)
q3_weeks = weekly.loc["2025-07":"2025-09"]
best_yoy_week = q3_weeks["revenue_yoy"].idxmax()
last_year_week = weekly.loc[best_yoy_week - pd.Timedelta(weeks=52)]
q3_weeks, best_yoy_week, last_year_week
```

```python
monthly["revenue_mom"] = monthly["revenue"].pct_change()
monthly["rate_change"] = monthly["conversion_rate"].diff()
monthly.loc["2025-07":"2025-09"]
```

```python
q3_yoy = daily.loc["2025-07":"2025-09", "revenue"].sum() / daily.loc["2024-07":"2024-09", "revenue"].sum() - 1
q3_yoy
```

</details>